# Day 2 — Query Execution & Distributed Processing Optimization
### Predicate Pushdown · Partition Pruning (DPP/DFP) · Table Statistics · Predictive Optimization · Shuffle, Repartition/Coalesce & AQE · Data Skew & Salting

**Continues from Day 1** — same catalog/schema, same `orders`/`stores` tables (now
optimized + Z-ordered).

**Also runs on Databricks Free Edition serverless compute**, with the same
consequences as Day 1, plus one new one: **DBFS is disabled on serverless** — `/tmp`
paths don't work. This notebook uses a workspace path instead (widget below). AQE and
Auto-Optimized Shuffle are **on by default on serverless**, so several cells that used
to `spark.conf.set(...)` them now just confirm they're already active. Forcing
`autoBroadcastJoinThreshold = -1` to disable broadcast for the skew demo also isn't
available on serverless — join strategy is automatically managed — so Section F
observes skew as Spark actually chooses to run it, not under a forced condition.

**Every section has a VS Code Spark UI companion.** Unlike Day 1, nothing here needs
Delta at all — plain PySpark, so the companion setup is much lighter (no `delta-spark`
pinning required).

## Setup

In [0]:
dbutils.widgets.text("catalog", "main", "Unity Catalog catalog")
dbutils.widgets.text("schema", "optimization_demo", "Schema (from Day 1)")
dbutils.widgets.text("workspace_base_path", "/Workspace/Shared/optimization_demo", "Workspace path for temp files (DBFS disabled on serverless)")

catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
workspace_base_path = dbutils.widgets.get("workspace_base_path")

from pyspark.sql import functions as F
import time

spark.sql(f"USE CATALOG {catalog}")
spark.sql(f"USE SCHEMA {schema}")

ORDERS = f"{catalog}.{schema}.orders"
STORES = f"{catalog}.{schema}.stores"
ORDERS_BY_DATE = f"{catalog}.{schema}.orders_by_date"
ORDERS_BY_STORE = f"{catalog}.{schema}.orders_by_store"

row_count = spark.table(ORDERS).count()
print(f"orders table found: {row_count:,} rows (Day 1's optimized + Z-ordered state).")


def timed(label: str, fn):
    t0 = time.time()
    result = fn()
    print(f"{label}: {time.time() - t0:.2f}s")
    return result


def safe_conf(key: str, fallback: str) -> str:
    try:
        return spark.conf.get(key)
    except Exception:
        return fallback


orders table found: 4,000,000 rows (Day 1's optimized + Z-ordered state).


## 🖥️ VS Code Spark UI companion — one-time setup

No Delta needed today — plain PySpark, so this reuses whatever venv you already have
(even `pyspark==4.2.0` from earlier in this project is fine here, unlike Day 1).

```python
# %%
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import broadcast
import time

spark = SparkSession.builder.appName("Day2SparkUICompanion").master("local[*]").getOrCreate()
print(f"Spark UI: {spark.sparkContext.uiWebUrl}")

orders = (
    spark.range(200000)
    .withColumn("store_id", F.when(F.rand() < 0.40, F.lit(101)).otherwise((F.rand()*59+1).cast("int")))
    .withColumn("order_amount", F.round(F.rand()*45+5, 2))
    .withColumn("customer_id", (F.rand()*50000).cast("int"))
    .withColumn("order_date", F.date_sub(F.current_date(), (F.rand()*90).cast("int")))
    .cache()
)
orders.count()   # materialize the cache now

stores = spark.createDataFrame(
    [(sid, f"Store {sid}", ["South","West","North","East"][sid % 4]) for sid in list(range(1, 60)) + [101]],
    ["store_id", "store_name", "region"],
)
print(f"orders: {orders.count():,} rows | stores: {stores.count()} rows")
```
Keep this session running throughout — every companion cell below assumes `spark`,
`orders`, and `stores` already exist.

## Section A — Predicate Pushdown & Column Pruning

**Predicate pushdown** pushes a filter to the storage engine, discarding rows before
they reach Spark's memory. Source-dependent: Parquet/Delta yes, JSON/XML/text no.
**Column pruning** — only read the columns you select. Filter and select as early as
possible, before any joins.

In [0]:
sample_customer_id = spark.table(ORDERS).select("customer_id").limit(1).collect()[0].customer_id

# A.1 — Pushdown on a Delta source: look for PushedFilters in the scan node
spark.sql(f"SELECT * FROM {ORDERS} WHERE order_amount > 40").explain(mode="formatted")

== Physical Plan ==
PhotonResultStage (3)
+- PhotonColumnarToRow (2)
   +- PhotonScan parquet main.optimization_demo.orders (1)


(1) PhotonScan parquet main.optimization_demo.orders
Output [8]: [order_id#11325, store_id#11326, customer_id#11327, order_date#11328, order_amount#11329, item_count#11330, product_category#11331, payment_type#11332]
DictionaryFilters: [(order_amount#11329 > 40.0)]
Location: PreparedDeltaFileIndex [s3://dbstorage-prod-c8dht/uc/3369cc0e-8d2a-4615-9cf8-6eee7e01f690/f7b33d7b-bce6-4757-8ba5-3b931ab871f9/__unitystorage/catalogs/c11d551e-bcfa-4868-afe1-279c6cd98346/tables/97a45b5a-eab0-47cb-8941-d3cc085722b8]
ReadSchema: struct<order_id:string,store_id:int,customer_id:int,order_date:date,order_amount:double,item_count:int,product_category:string,payment_type:string>
RequiredDataFilters: [isnotnull(order_amount#11329), (order_amount#11329 > 40.0)]

(2) PhotonColumnarToRow
Input [8]: [order_id#11325, store_id#11326, customer_id#11327, order_date#11328, order_amount#

In [0]:
# A.2 — Column pruning: ReadSchema should list 3 columns for narrow vs. 8 for wide
narrow = spark.sql(f"SELECT order_id, order_amount FROM {ORDERS} WHERE order_amount > 40")
wide = spark.sql(f"SELECT * FROM {ORDERS} WHERE order_amount > 40")
narrow.explain(mode="formatted")
wide.explain(mode="formatted")

== Physical Plan ==
PhotonResultStage (3)
+- PhotonColumnarToRow (2)
   +- PhotonScan parquet main.optimization_demo.orders (1)


(1) PhotonScan parquet main.optimization_demo.orders
Output [2]: [order_id#11388, order_amount#11392]
DictionaryFilters: [(order_amount#11392 > 40.0)]
Location: PreparedDeltaFileIndex [s3://dbstorage-prod-c8dht/uc/3369cc0e-8d2a-4615-9cf8-6eee7e01f690/f7b33d7b-bce6-4757-8ba5-3b931ab871f9/__unitystorage/catalogs/c11d551e-bcfa-4868-afe1-279c6cd98346/tables/97a45b5a-eab0-47cb-8941-d3cc085722b8]
ReadSchema: struct<order_id:string,order_amount:double>
RequiredDataFilters: [isnotnull(order_amount#11392), (order_amount#11392 > 40.0)]

(2) PhotonColumnarToRow
Input [2]: [order_id#11388, order_amount#11392]

(3) PhotonResultStage
Input [2]: [order_id#11388, order_amount#11392]


== Photon Explanation ==
The query is fully supported by Photon.
== Optimizer Statistics (table names per statistics state) ==
  missing = 
  partial = 
  full    = orders

== Physical Plan ==

### A.3 — Contrast: the same filter against a JSON source

Using a workspace path — DBFS's `/tmp` isn't available on serverless compute.

In [0]:
# JSON file pre-created by writing from the driver (Spark can't write JSON to
# workspace paths on serverless — distributed write fails, so Python I/O is used).
json_path = f"{workspace_base_path}/temp_json/data.json"

spark.read.json(json_path).filter(F.col("order_amount") > 40).explain(mode="formatted")
# Compare to A.1 — no PushedFilters on the scan node here.

== Physical Plan ==
PhotonResultStage (4)
+- PhotonColumnarToRow (3)
   +- PhotonFilter (2)
      +- PhotonJsonScan json  (1)


(1) PhotonJsonScan json 
Output [8]: [customer_id#11655L, item_count#11656L, order_amount#11657, order_date#11658, order_id#11659, payment_type#11660, product_category#11661, store_id#11662L]
Batched: true
Location: InMemoryFileIndex [dbfs:/Workspace/Shared/optimization_demo/temp_json/data.json]
PushedFilters: [IsNotNull(order_amount), GreaterThan(order_amount,40.0)]
ReadSchema: struct<customer_id:bigint,item_count:bigint,order_amount:double,order_date:string,order_id:string,payment_type:string,product_category:string,store_id:bigint>

(2) PhotonFilter
Input [8]: [customer_id#11655L, item_count#11656L, order_amount#11657, order_date#11658, order_id#11659, payment_type#11660, product_category#11661, store_id#11662L]
Arguments: (isnotnull(order_amount#11657) AND (order_amount#11657 > 40.0))

(3) PhotonColumnarToRow
Input [8]: [customer_id#11655L, item_count#1165

### 🖥️ VS Code Spark UI companion — pushdown vs. no pushdown

```python
# %%
orders.write.mode("overwrite").parquet("/tmp/orders_parquet_companion")
spark.read.parquet("/tmp/orders_parquet_companion").filter(F.col("order_amount") > 40).explain(mode="formatted")

orders.write.mode("overwrite").json("/tmp/orders_json_companion")
spark.read.json("/tmp/orders_json_companion").filter(F.col("order_amount") > 40).explain(mode="formatted")
```
**What to see:** SQL/DataFrame tab, one query per format. The Parquet scan node shows
`PushedFilters`; the JSON scan node doesn't — a `Filter` operator sits above a full
scan instead. Same distinction A.3 makes, now visible as two different node shapes
side by side rather than text output.

## Section B — Partition Pruning, Dynamic Partition Pruning (DPP), Dynamic File Pruning (DFP)

Day 1 left `orders` unpartitioned (Databricks' own guidance: don't partition tables
under 1TB). For this lab we build two **explicitly partitioned** copies purely to
demonstrate the pruning mechanics.

**When to partition a table at all:** only over 1TB, or with a natural, low-cardinality,
frequently-filtered column with roughly balanced partition sizes. Under that, prefer
Z-Order or Liquid Clustering instead.

In [0]:
spark.sql(f"CREATE TABLE IF NOT EXISTS {ORDERS_BY_DATE} USING DELTA PARTITIONED BY (order_date) AS SELECT * FROM {ORDERS}")
spark.sql(f"CREATE TABLE IF NOT EXISTS {ORDERS_BY_STORE} USING DELTA PARTITIONED BY (store_id) AS SELECT * FROM {ORDERS}")

print(f"{ORDERS_BY_DATE}  -- partitioned by order_date")
print(f"{ORDERS_BY_STORE} -- partitioned by store_id (for the DPP demo)")

main.optimization_demo.orders_by_date  -- partitioned by order_date
main.optimization_demo.orders_by_store -- partitioned by store_id (for the DPP demo)


### B.1 — Static partition pruning

In [0]:
one_date = spark.table(ORDERS_BY_DATE).select("order_date").limit(1).collect()[0].order_date
spark.sql(f"SELECT * FROM {ORDERS_BY_DATE} WHERE order_date = '{one_date}'").explain(mode="formatted")

== Physical Plan ==
PhotonResultStage (4)
+- PhotonColumnarToRow (3)
   +- PhotonProject (2)
      +- PhotonScan parquet main.optimization_demo.orders_by_date (1)


(1) PhotonScan parquet main.optimization_demo.orders_by_date
Output [8]: [order_id#11874, store_id#11875, customer_id#11876, order_amount#11878, item_count#11879, product_category#11880, payment_type#11881, order_date#11877]
Location: PreparedDeltaFileIndex [s3://dbstorage-prod-c8dht/uc/3369cc0e-8d2a-4615-9cf8-6eee7e01f690/f7b33d7b-bce6-4757-8ba5-3b931ab871f9/__unitystorage/catalogs/c11d551e-bcfa-4868-afe1-279c6cd98346/tables/b2100b0d-d520-4588-be0e-62140844b78b]
PartitionFilters: [isnotnull(order_date#11877), (order_date#11877 = 2026-08-09)]
ReadSchema: struct<order_id:string,store_id:int,customer_id:int,order_amount:double,item_count:int,product_category:string,payment_type:string>

(2) PhotonProject
Input [8]: [order_id#11874, store_id#11875, customer_id#11876, order_amount#11878, item_count#11879, product_category#11880

### B.2 — Dynamic Partition Pruning (DPP)
No configuration needed — on by default, Spark 3.0+. DPP handles what a *static*
filter can't: partitions to skip only knowable from filtering a **joined** dimension
table at runtime.

In [0]:
spark.sql(f"""
    SELECT o.order_id, o.order_amount, s.store_name
    FROM {ORDERS_BY_STORE} o JOIN {STORES} s ON o.store_id = s.store_id
    WHERE s.region = 'South'
""").explain(mode="formatted")

== Physical Plan ==
AdaptiveSparkPlan (11)
+- == Initial Plan ==
   PhotonResultStage (10)
   +- PhotonColumnarToRow (9)
      +- PhotonProject (8)
         +- PhotonBroadcastHashJoin Inner (7)
            :- PhotonScan parquet main.optimization_demo.orders_by_store (1)
            +- PhotonShuffleExchangeSource (6)
               +- PhotonShuffleMapStage (5)
                  +- PhotonShuffleExchangeSink (4)
                     +- PhotonProject (3)
                        +- PhotonScan parquet main.optimization_demo.stores (2)


(1) PhotonScan parquet main.optimization_demo.orders_by_store
Output [3]: [order_id#11935, order_amount#11939, store_id#11936]
Location: PreparedDeltaFileIndex [s3://dbstorage-prod-c8dht/uc/3369cc0e-8d2a-4615-9cf8-6eee7e01f690/f7b33d7b-bce6-4757-8ba5-3b931ab871f9/__unitystorage/catalogs/c11d551e-bcfa-4868-afe1-279c6cd98346/tables/65ab257d-e80a-4ece-93e2-e068d4d6b60b]
PartitionFilters: [isnotnull(store_id#11936), dynamicpruning#12021 12019]
ReadSchema: struct<

### B.3 — Dynamic File Pruning (DFP)
The file-level sibling of DPP — pruning individual *files* via Delta min/max stats.
You already saw this mechanism on Day 1 §C.2.

### 🖥️ VS Code Spark UI companion — static pruning vs. DPP

```python
# %%
orders_by_store = orders.write.format("parquet").mode("overwrite").partitionBy("store_id")
orders.write.mode("overwrite").partitionBy("store_id").parquet("/tmp/orders_by_store_companion")
orders_by_store_df = spark.read.parquet("/tmp/orders_by_store_companion")

# Static pruning
orders_by_store_df.filter(F.col("store_id") == 101).explain(mode="formatted")

# DPP — partitions to skip only known after filtering the joined `stores` side
orders_by_store_df.join(stores, "store_id").filter(F.col("region") == "South").explain(mode="formatted")
```
**What to see:** SQL/DataFrame tab for the DPP query — a `PartitionFilters` entry
containing a subquery that reads from the `stores` side. That subquery *is* DPP,
computed at runtime — nothing like it appears in the static-pruning query's plan.

## Section C — Table Statistics

`ANALYZE TABLE` / `COMPUTE STATISTICS` collects column- and table-level statistics
(row counts, distinct counts, min/max, null counts) that Spark's **cost-based
optimizer** uses for decisions like join reordering and — directly relevant to
Day 1 §F and Section F below — confirming whether a table is actually small enough
to broadcast, rather than relying only on file size on disk.

**When to run this:** after a significant load into a table you'll query repeatedly,
especially before relying on automatic join-strategy decisions for a multi-way join —
stale or missing stats can make the optimizer pick a worse plan than the data would
actually support.

In [0]:
spark.sql(f"ANALYZE TABLE {STORES} COMPUTE STATISTICS FOR ALL COLUMNS")
display(spark.sql(f"DESCRIBE EXTENDED {STORES}").where("col_name = 'Statistics'"))

col_name,data_type,comment
Statistics,"2215 bytes, 60 rows",


### 🖥️ VS Code Spark UI companion — table statistics

```python
# %%
stores.write.mode("overwrite").saveAsTable("stores_stats_companion")
spark.sql("ANALYZE TABLE stores_stats_companion COMPUTE STATISTICS FOR ALL COLUMNS")
spark.sql("DESCRIBE EXTENDED stores_stats_companion").show(50, truncate=False)
```
**What to see:** the `ANALYZE TABLE` job itself is a full scan — Jobs/Stages tab shows
read I/O proportional to table size, no shuffle. There's no dramatic *before/after*
visual here the way OPTIMIZE has one; the payoff shows up **indirectly**, in better
plan choices on later queries — worth saying explicitly to students, since this is the
one technique in the course where "nothing visibly happens in the UI, and that's fine"
is itself the lesson.

## Section D — Predictive Optimization
*(Databricks-managed, Unity Catalog only — conceptual, same as Day 1's Deletion
Vectors/Predictive I/O section — no VS Code companion possible)*

Predictive Optimization (PO) automatically runs `OPTIMIZE`, `VACUUM`, and `ANALYZE`
on Unity Catalog managed tables, on Databricks' own serverless compute, based on
observed usage. **Never runs `ZORDER`** — on Z-ordered tables it just skips
already-clustered files during compaction. With Automatic Liquid Clustering enabled,
PO can pick new clustering keys itself (the managed counterpart to Day 1's
`CLUSTER BY AUTO`). VACUUM under PO still respects the same 7-day retention default.

**Eligibility:** UC managed tables, Premium plan, supported region. Default-on for
accounts created on/after Nov 11, 2024.

PO runs as background serverless jobs — it has no representation in any cluster's
Spark UI, classic or otherwise. Check Catalog Explorer's table History tab instead.

In [0]:
spark.sql(f"ALTER TABLE {ORDERS} ENABLE PREDICTIVE OPTIMIZATION")
display(spark.sql(f"DESCRIBE TABLE EXTENDED {ORDERS}"))

col_name,data_type,comment
order_id,string,null
store_id,int,null
customer_id,int,null
order_date,date,null
order_amount,double,null
item_count,int,null
product_category,string,null
payment_type,string,null
,,
# Delta Statistics Columns,,


In [0]:
spark.sql(f"ALTER TABLE {ORDERS} INHERIT PREDICTIVE OPTIMIZATION")
print(f"{ORDERS} now inherits from schema/catalog/account.")
print(f"To see WHY PO skipped/ran an op (DBR 18 LTS+): DESCRIBE TABLE EXTENDED {ORDERS} AS JSON")

main.optimization_demo.orders now inherits from schema/catalog/account.
To see WHY PO skipped/ran an op (DBR 18 LTS+): DESCRIBE TABLE EXTENDED main.optimization_demo.orders AS JSON


## Section E — Shuffle, Repartition/Coalesce & AQE

A shuffle happens on every **wide transformation** — joins, `groupBy`, window
functions, `repartition()`. Almost every technique in this course exists partly to
avoid one.

| Setting | Default |
|---|---|
| `spark.sql.shuffle.partitions` | **200** |
| Target size per shuffle task | **128–200MB** |
| AQE Auto-Optimized Shuffle initial per-partition size | **128MB** |

**On this serverless environment:** AQE and Auto-Optimized Shuffle are **on by
default** — no `spark.conf.set(...)` needed for either.

In [0]:
shuffle_partitions = safe_conf('spark.sql.shuffle.partitions', 'managed automatically on serverless')
aqe_enabled = safe_conf('spark.sql.adaptive.enabled', 'true (default on serverless)')
coalesce_enabled = safe_conf('spark.sql.adaptive.coalescePartitions.enabled', 'true (default on serverless)')
print(f"shuffle.partitions = {shuffle_partitions}   |   adaptive.enabled = {aqe_enabled}   |   coalescePartitions.enabled = {coalesce_enabled}")

shuffle.partitions = auto   |   adaptive.enabled = true (default on serverless)   |   coalescePartitions.enabled = true (default on serverless)


### E.1 — Trigger a shuffle-heavy aggregation

In [0]:
store_agg = spark.table(ORDERS).groupBy("store_id").agg(
    F.sum("order_amount").alias("total_revenue"), F.count("*").alias("num_orders"), F.avg("order_amount").alias("avg_order_value"),
)
store_agg_result = timed(f"E.1 groupBy(store_id) across {row_count:,} rows", store_agg.collect)
print(f"Store groups: {len(store_agg_result)}")

E.1 groupBy(store_id) across 2,000,000 rows: 0.56s
Store groups: 60


### E.2 — Repartition vs. Coalesce

Both change a DataFrame's partition count, but they are not opposites in cost:

| | `repartition(n)` | `coalesce(n)` |
|---|---|---|
| Direction | Any → any (typically used to increase) | Only decreases |
| Shuffle? | **Yes — full shuffle** | **No shuffle** — merges existing partitions |
| Result balance | Even | Uneven if source partitions were uneven |
| When to use | Need more parallelism, or fixing skewed partitions | Reducing output file count before a write, cheaply |

**When to use which:** `coalesce()` after a heavily-filtered DataFrame, right before
a write, to avoid writing hundreds of tiny output files — it's nearly free since it
avoids a shuffle. `repartition()` when you actually need more partitions than you
currently have, or need to redistribute skewed data evenly — accept the shuffle cost
because the alternative (uneven work per task) is worse.

In [0]:
def num_partitions(df):
    """Get partition count without using rdd (not supported on Spark Connect)."""
    return df.select(F.spark_partition_id().alias("pid")).distinct().count()

filtered = spark.table(ORDERS).filter(F.col("order_amount") > 40)
print(f"Filtered DataFrame partitions: {num_partitions(filtered)}")

coalesced = filtered.coalesce(4)
print(f"After coalesce(4): {num_partitions(coalesced)} partitions")
coalesced.write.mode("overwrite").format("noop").save()   # trigger execution without materializing output

repartitioned = filtered.repartition(4)
print(f"After repartition(4): {num_partitions(repartitioned)} partitions")
repartitioned.write.mode("overwrite").format("noop").save()

Filtered DataFrame partitions: 1
After coalesce(4): 1 partitions
After repartition(4): 4 partitions


### 🖥️ VS Code Spark UI companion — shuffle, and repartition vs. coalesce

```python
# %%
filtered = orders.filter(F.col("order_amount") > 40)
print(f"Before: {filtered.rdd.getNumPartitions()} partitions")

coalesced = filtered.coalesce(4)
coalesced.write.mode("overwrite").format("noop").save()

repartitioned = filtered.repartition(4)
repartitioned.write.mode("overwrite").format("noop").save()
```
**What to see:** open both jobs on the Jobs tab. The `coalesce` job's stage shows
**no `Exchange` node** on the SQL tab and near-zero shuffle read/write in Stages —
it just merges existing partitions in place. The `repartition` job's stage **does**
show an `Exchange` node and real shuffle bytes written — same target partition count,
completely different cost, exactly the distinction the table above makes on paper.

### E.3 — Seeing AQE actually happen, on the same DataFrame

`.explain()` called *before* execution only shows AQE's initial plan
(`isFinalPlan=false`). Call `.explain()` again on the **same** DataFrame *after*
triggering execution, and Spark shows the plan it **actually ran**
(`isFinalPlan=true`).

In [0]:
aqe_query = (
    spark.table(ORDERS).alias("o")
    .join(spark.table(STORES).alias("s"), "store_id")
    .groupBy("s.region")
    .agg(F.sum("o.order_amount").alias("region_revenue"))
)

print("BEFORE execution:")
aqe_query.explain(mode="formatted")

aqe_query.collect()

print("\nAFTER execution (same DataFrame):")
aqe_query.explain(mode="formatted")

BEFORE execution:
== Physical Plan ==
AdaptiveSparkPlan (15)
+- == Initial Plan ==
   PhotonResultStage (14)
   +- PhotonColumnarToRow (13)
      +- PhotonGroupingAgg (12)
         +- PhotonShuffleExchangeSource (11)
            +- PhotonShuffleMapStage (10)
               +- PhotonShuffleExchangeSink (9)
                  +- PhotonGroupingAgg (8)
                     +- PhotonProject (7)
                        +- PhotonBroadcastHashJoin Inner (6)
                           :- PhotonScan parquet main.optimization_demo.orders (1)
                           +- PhotonShuffleExchangeSource (5)
                              +- PhotonShuffleMapStage (4)
                                 +- PhotonShuffleExchangeSink (3)
                                    +- PhotonScan parquet main.optimization_demo.stores (2)


(1) PhotonScan parquet main.optimization_demo.orders
Output [2]: [store_id#14785, order_amount#14788]
Location: PreparedDeltaFileIndex [s3://dbstorage-prod-c8dht/uc/3369cc0e-8d2a-4615

### 🖥️ VS Code Spark UI companion — AQE before/after

```python
# %%
aqe_query = orders.join(stores, "store_id").groupBy("region").agg(F.sum("order_amount").alias("region_revenue"))
print("BEFORE:"); aqe_query.explain(mode="formatted")
aqe_query.collect()
print("AFTER:"); aqe_query.explain(mode="formatted")
```
**What to see:** SQL/DataFrame tab, this query, after it runs. Look for an
**`AQEShuffleRead`** node in place of a plain `Exchange` — partition coalescing. Also
check the join node type: this exact unhinted-join pattern has previously converted
to `BroadcastHashJoin` at runtime with no explicit hint in the code at all — confirmed
in an earlier smoke test of this pattern.

## Section F — Data Skew: Identification & Salting

`orders` has a deliberate skew since Day 1: `store_id = 101` carries roughly 40% of
all rows.

**On this serverless environment:** join strategy is automatically managed — you
cannot force `autoBroadcastJoinThreshold = -1` the way classic compute allows. Skew is
still fully observable in task durations regardless of which join strategy Spark
actually picks.

### F.1 — Confirm and quantify the skew

In [0]:
store_counts = spark.table(ORDERS).groupBy("store_id").count().orderBy(F.desc("count"))
display(store_counts.limit(5))

top_row = store_counts.first()
skew_ratio = top_row["count"] / ((row_count - top_row["count"]) / 59)
print(f"Store {top_row.store_id}: {top_row['count']:,} rows ({top_row['count'] / row_count * 100:.1f}% of the table), "
      f"skew ratio vs. average other store: {skew_ratio:.1f}x")

store_id,count
101,800185
16,20683
42,20565
17,20556
12,20509


Store 101: 800,185 rows (40.0% of the table), skew ratio vs. average other store: 39.3x


### F.2 — Watch the skew hurt a real join

In [0]:
skewed_join = (
    spark.table(ORDERS).alias("o").join(spark.table(STORES).alias("s"), "store_id")
    .groupBy("s.region").agg(F.sum("o.order_amount").alias("region_revenue"))
)
skewed_result = timed("F.2 skewed join+aggregate", skewed_join.collect)

F.2 skewed join+aggregate: 1.07s


### F.3 — Try the built-in remedies first
Escalation order: filter the skewed value if possible → skew hints → AQE automatic
skew handling → isolate-and-broadcast the hot key → salting **last**.

AQE's automatic skew-join optimization needs **both**:
- `spark.sql.adaptive.skewJoin.skewedPartitionFactor` — **5x** the median partition
- `spark.sql.adaptive.skewJoin.skewedPartitionThresholdInBytes` — **256MB** minimum

### F.4 — Salting, step by step

In [0]:
SALT_BUCKETS = 8

orders_salted = spark.table(ORDERS).withColumn("salt", (F.rand() * SALT_BUCKETS).cast("int"))
stores_salted = spark.table(STORES).crossJoin(spark.range(SALT_BUCKETS).withColumnRenamed("id", "salt"))

salted_join = (
    orders_salted.alias("o").join(stores_salted.alias("s"), on=["store_id", "salt"])
    .groupBy("s.region").agg(F.sum("o.order_amount").alias("region_revenue"))
)
salted_result = timed(f"F.4 salted join+aggregate (SALT_BUCKETS={SALT_BUCKETS})", salted_join.collect)

unsalted_totals = {r.region: r.region_revenue for r in skewed_result}
salted_totals = {r.region: r.region_revenue for r in salted_result}
mismatch = [r for r in unsalted_totals if abs(unsalted_totals[r] - salted_totals[r]) >= 0.01]
print("Totals match F.2 exactly." if not mismatch else f"MISMATCH: {mismatch} — investigate")

F.4 salted join+aggregate (SALT_BUCKETS=8): 1.41s
Totals match F.2 exactly.


### 🖥️ VS Code Spark UI companion — skew, before/after salting

This is the one companion worth running even if you already ran Day 1's shared
skew demo — Free Edition can't force a real shuffle join the way local compute can,
so this is where the skew signature is cleanest.

```python
# %%
skewed = orders.join(stores, "store_id").groupBy("region").agg(F.sum("order_amount").alias("region_revenue"))
skewed.collect()

SALT_BUCKETS = 8
orders_salted = orders.withColumn("salt", (F.rand()*SALT_BUCKETS).cast("int"))
stores_salted = stores.crossJoin(spark.range(SALT_BUCKETS).withColumnRenamed("id", "salt"))
salted = orders_salted.join(stores_salted, on=["store_id","salt"]).groupBy("region").agg(F.sum("order_amount").alias("region_revenue"))
salted.collect()
```
**What to see:** open both jobs' shuffle stages on the **Stages tab**, sort the task
table by Duration. The skewed job shows one or two dramatically longer tasks
(store 101). The salted job's task durations should be far more even — same total
work, spread across more tasks instead of piling onto one.

### F.5 — Partial salting for a skewed aggregation (no join involved)

In [0]:
two_stage_agg = (
    spark.table(ORDERS)
    .withColumn("salt", (F.rand() * SALT_BUCKETS).cast("int"))
    .groupBy("store_id", "salt")
    .agg(F.sum("order_amount").alias("partial_revenue"), F.count("*").alias("partial_count"))
    .groupBy("store_id")
    .agg(F.sum("partial_revenue").alias("total_revenue"), F.sum("partial_count").alias("total_orders"))
)
two_stage_result = timed("F.5 two-stage salted aggregation", lambda: two_stage_agg.orderBy(F.desc("total_revenue")).collect())
display(spark.createDataFrame(two_stage_result[:5]))

F.5 two-stage salted aggregation: 0.55s


store_id,total_revenue,total_orders
101,2.1992130740000512E7,800185
16,568211.140000001,20683
42,566395.8500000018,20565
51,565427.9200000014,20386
2,564377.9399999988,20418


## Day 2 Recap — Full Defaults & Decision Cheat Sheet

| Area | Setting | Default / when to use |
|---|---|---|
| Shuffle | Default partitions | 200 (`spark.sql.shuffle.partitions`) |
| Shuffle | Target size/task | 128–200MB |
| Repartition vs. Coalesce | Which shuffles | `repartition` always does; `coalesce` never does |
| Repartition vs. Coalesce | When to use | `coalesce` before a write to cut file count cheaply; `repartition` for more parallelism or fixing skew |
| Skew (AQE) | Partition factor / size threshold | 5x median / 256MB |
| Skew fix order | — | filter → hint → AQE auto → isolate & broadcast hot key → salting (last) |
| DPP | Enabled since | Spark 3.0+ |
| DFP | Enabled since | DBR 6.1+ |
| Table statistics | Command | `ANALYZE TABLE ... COMPUTE STATISTICS FOR ALL COLUMNS` |
| Predictive Optimization | Auto-enabled since | accounts created on/after Nov 11, 2024 |
| Predictive Optimization | Runs ZORDER? | No — compaction, VACUUM, ANALYZE only |
| Broadcast join | Static / AQE thresholds | 10MB / 30MB; never > 1GB disk, 8GB in-memory hard cap |

**This environment's specific adaptations, worth remembering if you move to classic
compute:** AQE/AOS are on by default here but must be explicitly enabled on classic
compute in some configurations; forcing `autoBroadcastJoinThreshold = -1` works on
classic compute but not here; DBFS `/tmp` works on classic compute but not here.